In [1]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms

from torch.utils.data import Dataset, DataLoader
from torchvision import models

In [3]:
PATH_OR = os.path.join('..', 'content', 'mnist_original.mat')
PATH_AD = os.path.join('..', 'content', 'adversarial_data_1000.mat')

In [4]:
mnist = loadmat(PATH_OR)
adverserial = loadmat(PATH_AD)

In [5]:
TRAIN_RATIO = 0.9
EPOCHS = 10

In [6]:
mnist_data = mnist['data'].T
mnist_labels = mnist['label'].reshape(-1)

adversarial_data = adverserial['adversarial_examples']
adversarial_labels = adverserial['labels'].reshape(-1)

In [7]:
if mnist_data.dtype != adversarial_data.dtype:
    mnist_data = mnist_data.astype(adversarial_data.dtype)

combined_data = np.concatenate((mnist_data, adversarial_data), axis=0)
combined_labels = np.concatenate((mnist_labels, adversarial_labels), axis=0)

In [8]:
train_mnist_data, test_mnist_data = mnist_data[:int(len(mnist_data) * TRAIN_RATIO)], mnist_data[int(len(mnist_data) * TRAIN_RATIO):]
train_mnist_labels, test_mnist_labels = mnist_labels[:int(len(mnist_labels) * TRAIN_RATIO)], mnist_labels[int(len(mnist_labels) * TRAIN_RATIO):]

train_adversarial_data, test_adversarial_data = adversarial_data[:int(len(adversarial_data) * TRAIN_RATIO)], adversarial_data[int(len(adversarial_data) * TRAIN_RATIO):]
train_adversarial_labels, test_adversarial_labels = adversarial_labels[:int(len(adversarial_labels) * TRAIN_RATIO)], adversarial_labels[int(len(adversarial_labels) * TRAIN_RATIO):]

train_combined_data, test_combined_data = np.concatenate((train_mnist_data, train_adversarial_data), axis=0), np.concatenate((test_mnist_data, test_adversarial_data), axis=0)
train_combined_labels, test_combined_labels = np.concatenate((train_mnist_labels, train_adversarial_labels), axis=0), np.concatenate((test_mnist_labels, test_adversarial_labels), axis=0)

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [10]:
class MNISTDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx].reshape(28, 28).astype(np.uint8)
        if self.transform:
            img = self.transform(img)
        label = int(self.labels[idx])
        return img, label

In [11]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

In [20]:
train_dataset = MNISTDataset(train_mnist_data, train_mnist_labels, transform=transform)
test_dataset = MNISTDataset(test_mnist_data, test_mnist_labels, transform=transform)

adverserial_dataset = MNISTDataset(adversarial_data, adversarial_labels, transform=transform)

In [21]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [22]:
adverserial_dataset = MNISTDataset(train_adversarial_data, train_adversarial_labels, transform=transform)
adverserial_loader = DataLoader(adverserial_dataset, batch_size=32, shuffle=True)

In [15]:
model_or = models.resnet18(weights=None)
model_or.fc = nn.Linear(model_or.fc.in_features, 10)
model_or = model_or.to(device)

In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_or.parameters(), lr=0.001)

# Normal Training

In [17]:
for epoch in range(EPOCHS):
    model_or.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_or(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

In [18]:
resnet18_or_model = os.path.join('..', 'models', 'resnet18_mnist_surrogate_or.pth')
torch.save(model_or.state_dict(), resnet18_or_model)

In [19]:
model_or.load_state_dict(torch.load(resnet18_or_model))

model_or.eval()
correct_or, correct_ad = 0, 0
total_or, total_ad = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_or(images)
        _, predicted = torch.max(outputs.data, 1)
        total_or += labels.size(0)
        correct_or += (predicted == labels).sum().item()
print(f"Original Accuracy: {100 * correct_or / total_or:.2f}%")

with torch.no_grad():
    for images, labels in adverserial_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_or(images)
        _, predicted = torch.max(outputs.data, 1)
        total_ad += labels.size(0)
        correct_ad += (predicted == labels).sum().item()
print(f"Adverserial Accuracy: {100 * correct_ad / total_ad:.2f}%")

print(f"Overall Accuracy: {100 * (correct_or + correct_ad) / (total_or + total_ad):.2f}%")

Original Accuracy: 99.36%
Adverserial Accuracy: 12.00%
Overall Accuracy: 89.41%


# Adverserial Training

In [12]:
train_dataset = MNISTDataset(train_combined_data, train_combined_labels, transform=transform)
test_dataset = MNISTDataset(test_combined_data, test_combined_labels, transform=transform)

In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [14]:
model_ad = models.resnet18(weights=None)
model_ad.fc = nn.Linear(model_ad.fc.in_features, 10)
model_ad = model_ad.to(device)

In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_ad.parameters(), lr=0.001)

In [17]:
for epoch in range(EPOCHS):
    model_ad.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_ad(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

In [18]:
resnet18_ad_model = os.path.join('..', 'models', 'resnet18_mnist_surrogate_ad.pth')
torch.save(model_ad.state_dict(), resnet18_ad_model)

In [23]:
model_ad.load_state_dict(torch.load(resnet18_ad_model))

model_ad.eval()
correct_or, correct_ad = 0, 0
total_or, total_ad = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_ad(images)
        _, predicted = torch.max(outputs.data, 1)
        total_or += labels.size(0)
        correct_or += (predicted == labels).sum().item()
print(f"Original Accuracy: {100 * correct_or / total_or:.2f}%")

with torch.no_grad():
    for images, labels in adverserial_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_ad(images)
        _, predicted = torch.max(outputs.data, 1)
        total_ad += labels.size(0)
        correct_ad += (predicted == labels).sum().item()
print(f"Adverserial Accuracy: {100 * correct_ad / total_ad:.2f}%")

print(f"Overall Accuracy: {100 * (correct_or + correct_ad) / (total_or + total_ad):.2f}%")

Original Accuracy: 99.41%
Adverserial Accuracy: 12.00%
Overall Accuracy: 89.46%
